In [ ]:
import sys, os
sys.path.append(os.path.abspath("../.."))

In [ ]:
import numpy as np
import pandas as pd
from preprocessing.preprocess import prep, append_results, eval_thresholds
from preprocessing.target import ttp_target, hybrid_target
from metrics.Metrics import merged_metrics
name = "AFKS"
df: pd.DataFrame = pd.read_csv(f"/Users/side/Desktop/Trading Chaos AI/df/clean_df/{name}.csv")

In [ ]:
class DeepLOBLike(nn.Module):
    def init(self, n_features, n_classes=3, hidden=64, dropout=0.2):
        super().init()
        # x: (B,L,F) -> (B,1,L,F) как "картинка"
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=(3, 1), padding=(1,0)),
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=(3, 1), padding=(1,0)),
            nn.ReLU(),
        )
        # "inception" по времени
        self.incept_a = nn.Conv2d(32, 32, kernel_size=(1,1))
        self.incept_b = nn.Conv2d(32, 32, kernel_size=(3,1), padding=(1,0))
        self.incept_c = nn.Conv2d(32, 32, kernel_size=(5,1), padding=(2,0))

        self.pool = nn.AdaptiveAvgPool2d((None, 1))   # сжать F до 1
        self.lstm = nn.LSTM(input_size=96, hidden_size=hidden, batch_first=True)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden, n_classes)

    def forward(self, x):
        b,l,f = x.shape
        x = x.unsqueeze(1)         # (B,1,L,F)
        x = self.conv1(x)          # (B,32,L,F)
        a = torch.relu(self.incept_a(x))
        b2 = torch.relu(self.incept_b(x))
        c = torch.relu(self.incept_c(x))
        x = torch.cat([a,b2,c], dim=1)   # (B,96,L,F)
        x = self.pool(x).squeeze(-1)     # (B,96,L)
        x = x.transpose(1,2)             # (B,L,96)
        out,_ = self.lstm(x)             # (B,L,H)
        h = self.drop(out[:, -1, :])
        return self.fc(h)

def train_deeplob_ttp(df, train_size, test_size, step, seq_len=128):
    set_seed(42)
    splitter = prep(
        df=df,
        target_fn=ttp_target, target_name="ttp", target_col="TTP_class",
        horizons=[12,24,48],
        train_size=train_size, test_size=test_size, step=step,
        target_kwargs={"n_classes":3},
        scale_cols=[
            "Open","High","Low","Close",
            "Alligator_Jaw","Alligator_Teeth","Alligator_Lips",
            "AO","AddOn_Anchor_Level","AddOn_Size_Pct"
        ]
    )

    for X_train, X_test, y_train, y_test, scaler in splitter:
        tr_ds = SeqDataset(X_train, y_train, seq_len)
        te_ds = SeqDataset(X_test,  y_test,  seq_len)
        tr_loader = DataLoader(tr_ds, batch_size=128, shuffle=True)
        te_loader = DataLoader(te_ds, batch_size=256, shuffle=False)

        model = DeepLOBLike(n_features=X_train.shape[1], n_classes=3, hidden=64)

        y_pred = train_torch_classifier(model, tr_loader, te_loader, epochs=20, lr=1e-3)
        y_test_seq = y_test[seq_len-1:]
        metrics = merged_metrics(y_test_seq, y_pred)

        append_results({
            "task_type":"classification",
            "model_name":"DeepLOB-like",
            "model_family":"deep",
            "model_params":{"hidden":64,"seq_len":seq_len},
            "target_name":"ttp","target_variant":"3class","horizons":"12_24_48",
            **metrics
        })